In [ ]:
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(8)
!ollama pull llama3.1:8b-instruct-q4_K_M
print("Ollama ready.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Ollama ready.


In [ ]:
!pip install -q requests

In [ ]:
import zipfile
import json
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import os
import time
import random
import requests
from google.colab import userdata
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import re

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
#checking whether ollama is running
resp = requests.get("http://localhost:11434/api/tags")
print("Ollama status:", resp.status_code)
models = [m["name"] for m in resp.json().get("models", [])]
print("Available models:", models)

Ollama status: 200
Available models: ['llama3.1:8b-instruct-q4_K_M']


In [ ]:
OLLAMA_MODEL = "llama3.1:8b-instruct-q4_K_M"
OLLAMA_URL = "http://localhost:11434/api/chat"

In [ ]:
LABEL_DESC = "aapd_label_descriptions_prompt_friendly.json"
VAL_GREEDY_INPUT = "aapd_validation_greedy_llm_input_data.jsonl"
VAL_TOPK_INPUT = "aapd_validation_top_k_llm_input_data.jsonl"
VAL_GOLD_LABELS = "aapd_validation_gold_labels.json"
N_RETRIEVED = 1

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/thesis_results/AAPD_LLAMA_OLLAMA_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GREEDY_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_greedy_predictions.jsonl"
TOPK_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_topk_predictions.jsonl"
LABELS_ONLY_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_labels_only_predictions.jsonl"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [ ]:
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

print("Validation shape:", aapd_df_val.shape)
print("Test shape:", aapd_df_test.shape)
#ok

Validation shape: (1000, 3)
Test shape: (1000, 3)


In [ ]:
#loading retrieval results

#helper function
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

#validation only for now
val_greedy_items = load_jsonl(VAL_GREEDY_INPUT)
val_topk_items = load_jsonl(VAL_TOPK_INPUT)

print(val_greedy_items[0].keys())
print(val_greedy_items[0])

dict_keys(['query_id', 'target_text', 'retrieved_examples'])
{'query_id': 0, 'target_text': 'linked open datasets about scholarly publications enable the development and integration of sophisticated end user services however , richer datasets are still needed the first goal of this challenge was to investigate novel approaches to obtain such semantic data in particular , we were seeking methods and tools to extract information from scholarly publications , to publish it as lod , and to use queries over this lod to assess quality this year we focused on the quality of workshop proceedings , and of journal articles w r t their citation network a third , open task , asked to showcase how such semantic data could be exploited and how semantic web technologies could help in this emerging context\n', 'retrieved_examples': [{'train_index': 26088, 'labels': ['Databases', 'Computational Engineering, Finance, and Science', 'Information Retrieval'], 'text': 'the linked clinical trials \\( linkedc

In [ ]:
#for evaluating the validation results
with open(VAL_GOLD_LABELS, "r", encoding="utf-8") as f:
    val_gold_labels = json.load(f)

In [ ]:
#reusing the same mlb I created for SVC
mlb = joblib.load("mlb.joblib")
ALL_LABELS = list(mlb.classes_)
print("Number of labels:", len(ALL_LABELS))

#label block for inserting into prompt
with open(LABEL_DESC, "r", encoding="utf-8") as f:
    label_descriptions = json.load(f)

#check that descriptions and mlb labels match
desc_labels = set(label_descriptions.keys())
mlb_labels = set(ALL_LABELS)
assert mlb_labels == desc_labels, "Mismatch!"

Number of labels: 54


In [ ]:
OLLAMA_OPTIONS = {
    "temperature": 0,
    "num_predict": 192,
    "seed": SEED}

In [ ]:
def label_block(label_descriptions, labels):
    lines = []
    for label in labels:
        desc = label_descriptions[label]
        lines.append(f" - {label}: {desc}")
    return "\n".join(lines)

LABEL_BLOCK = label_block(label_descriptions, ALL_LABELS)

print(LABEL_BLOCK)
#will be provided in-context
print("Number of labels:", len(ALL_LABELS)) #ok
print(f"Number of words:{len(LABEL_BLOCK.split())}")

 - Adaptation and Self-Organizing Systems: Focuses on systems that adapt and self-organize, including statistical physics and stochastic processes. Does not cover general machine learning topics outside of self-organization. Relevant to physics applications in complex systems.
 - Applications: Covers the application of statistical methods across various fields such as biology, engineering, and social sciences. Does not include theoretical statistics or methodology. Emphasizes practical implementations and case studies.
 - Artificial Intelligence: Encompasses all AI topics except for Vision, Robotics, Machine Learning, Multiagent Systems, and Computation and Language. Does not include practical applications of AI methods, which may fall under Applications. Focuses on theoretical aspects like knowledge representation and planning.
 - Combinatorics: Covers discrete mathematics topics such as graph theory, enumeration, and combinatorial optimization. Does not include continuous mathematics

In [ ]:
label_schema = {
    "type": "object",
    "properties": {
        "predicted_labels": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ALL_LABELS},
        }
    },
    "required": ["predicted_labels"],
    "additionalProperties": False}


In [ ]:
def warmup_ollama():
    """Send a short request to load the model into GPU memory."""
    resp = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": "Hi"}],
        "stream": False,
        "options": {"num_predict": 5}
    })
    print("Ollama warmup done:", resp.status_code)

In [ ]:
warmup_ollama()

Ollama warmup done: 200


In [ ]:
#check
print("Validation df length:", len(aapd_df_val))
print("Greedy items length:", len(val_greedy_items))
print("Top-k items length:", len(val_topk_items))
#checking whether the first target text is the same as val
print(val_greedy_items[0]["target_text"][:200])
print(aapd_df_val.iloc[0]["text"][:200])


Validation df length: 1000
Greedy items length: 1000
Top-k items length: 1000
linked open datasets about scholarly publications enable the development and integration of sophisticated end user services however , richer datasets are still needed the first goal of this challenge 
linked open datasets about scholarly publications enable the development and integration of sophisticated end user services however , richer datasets are still needed the first goal of this challenge 


In [ ]:
#since the AAPD abstracts are preprocessed in a way that punctuation appears after spaces (making them separate tokens),
#which is useful for SVC/transformers but not necessarily for LLMs
#I will send the text as more natural looking
def detokenize(text):
    text = str(text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\s+", " ", text)
    return text

#formatting the retrieved examples for in-prompt placement
def format_retrieved_examples(retrieved_examples, n=N_RETRIEVED):
    example_blocks = []
    for i, ex in enumerate(retrieved_examples[:n], start=1):
        text = detokenize(ex["text"])
        labels = ex["labels"]

        example_blocks.append(
            f"Example {i}\n"
            f"Abstract:\n{text}\n"
            f"Labels:\n{json.dumps(labels, ensure_ascii=False)}")
    return "\n\n".join(example_blocks)


##Building prompts

In [ ]:
def build_prompt_with_examples(target_text, retrieved_examples):
    target_text = detokenize(target_text)
    examples_block = format_retrieved_examples(retrieved_examples)

    system_message = (
        "You are an expert in multi-label topic classification. "
        "Use only the allowed label names and return only the required JSON object.")

    user_message = f"""
You are performing multi-label topic classification for academic abstracts.
Your task is to assign the most appropriate topic labels to the target abstract.
Use only labels from the allowed label set below.

Allowed labels and descriptions:
{LABEL_BLOCK}

Below are retrieved labelled demonstrations. They show examples of abstracts and their assigned labels.
{examples_block}

Now classify the target abstract.

Target abstract:
{target_text}

Return exactly one JSON object in this format:
{{"predicted_labels": ["label1", "label2"]}}

Rules:
- Use only exact label names from the allowed label set.
- Select only labels that are directly supported by the main topic of the target abstract.
- Use the retrieved demonstrations as guidance, but do not copy their labels unless the target abstract itself supports them.
- Return only the JSON object, with no explanation or extra text.

""".strip()

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}]

In [ ]:
def build_prompt_labels_only(target_text):
    target_text = detokenize(target_text)

    system_message = (
        "You are an expert in multi-label topic classification. "
        "Use only the allowed label names and return only the required JSON object.")

    user_message = f"""
You are performing multi-label topic classification for academic abstracts.
Your task is to assign the most appropriate topic labels to the target abstract.
Use only labels from the allowed label set below.

Allowed labels and descriptions:
{LABEL_BLOCK}

Now classify the target abstract.

Target abstract:
{target_text}

Return exactly one JSON object in this format:
{{"predicted_labels": ["label1", "label2"]}}

Rules:
- Use only exact label names from the allowed label set.
- Select only labels that are directly supported by the main topic of the target abstract.
- Return only the JSON object, with no explanation or extra text.

""".strip()

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}]

In [ ]:
def build_greedy_messages(item):
    return build_prompt_with_examples(target_text=item["target_text"], retrieved_examples=item["retrieved_examples"])


def build_topk_messages(item):
    return build_prompt_with_examples(target_text=item["target_text"], retrieved_examples=item["retrieved_examples"])


def build_labels_only_messages(item):
    return build_prompt_labels_only(target_text=item["target_text"])

In [ ]:
def generate_ollama_response(messages):
    total_start = time.time()

    resp = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": messages,
        "format": label_schema,
        "stream": False,
        "options": OLLAMA_OPTIONS
    })
    resp.raise_for_status()
    data = resp.json()

    total_runtime = time.time() - total_start
    raw_output = data["message"]["content"].strip()

    # Ollama returns token counts and timing natively
    input_tokens = data.get("prompt_eval_count")
    output_tokens = data.get("eval_count")
    # eval_duration is in nanoseconds
    eval_ns = data.get("eval_duration", 0)
    generation_runtime = eval_ns / 1e9

    return {
        "response": raw_output,
        "generation_runtime_seconds": generation_runtime,
        "total_sample_runtime_seconds": total_runtime,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }

In [ ]:
def parse_schema_output(raw_output):
    raw_output = str(raw_output).strip()
    parsed = json.loads(raw_output)

    labels = parsed["predicted_labels"]

    if not isinstance(labels, list):
        raise ValueError(f"predicted_labels is not a list: {raw_output}")

    invalid_labels = [
        label for label in labels
        if label not in ALL_LABELS
    ]

    if invalid_labels:
        raise ValueError(f"Invalid labels: {invalid_labels}")

    valid_labels = []
    for label in labels:
        if label not in valid_labels:
            valid_labels.append(label)

    return {
        "predicted_labels": valid_labels,
        "raw_output": raw_output}

In [ ]:
def get_done_query_ids(output_path):
    done = set()

    if not os.path.exists(output_path):
        return done

    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    row = json.loads(line)
                    if row.get("status") == "ok":
                        done.add(row["query_id"])
                except Exception:
                    pass
    return done

In [ ]:
def run_ollama_validation_checkpointed(
    items,
    output_path,
    strategy_name,
    build_messages_fn,
    gold_labels_dict,
    max_items=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if max_items is not None:
        items = items[:max_items]

    done_query_ids = get_done_query_ids(output_path)

    remaining_items = [
        item for item in items
        if item["query_id"] not in done_query_ids]

    print(f"Already completed: {len(done_query_ids)}")
    print(f"Remaining to run: {len(remaining_items)}")

    experiment_start = time.time()

    for item in remaining_items:
        query_id = item["query_id"]
        sample_start = time.time()

        try:
            messages = build_messages_fn(item)

            generation = generate_ollama_response(messages)
            parsed = parse_schema_output(generation["response"])

            row = {
                "status": "ok",
                "strategy": strategy_name,
                "query_id": query_id,
                "gold_labels": gold_labels_dict[str(query_id)],
                "predicted_labels": parsed["predicted_labels"],
                "raw_output": parsed["raw_output"],
                "input_tokens": generation["input_tokens"],
                "output_tokens": generation["output_tokens"],
                "generation_runtime_seconds": generation["generation_runtime_seconds"],
                "total_sample_runtime_seconds": generation["total_sample_runtime_seconds"],
                "wall_time_seconds": time.time() - sample_start,
                "timestamp": datetime.now().isoformat()}

        except Exception as e:
            row = {
                "status": "error",
                "strategy": strategy_name,
                "query_id": query_id,
                "error": repr(e),
                "wall_time_seconds": time.time() - sample_start,
                "timestamp": datetime.now().isoformat()}

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"Finished. Total wall time: {(time.time() - experiment_start) / 60:.2f} minutes")

In [ ]:
def load_prediction_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                if row.get("status") == "ok":
                    rows.append(row)

    return rows

def load_prediction_jsonl_all(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


def evaluate_prediction_file(path, mlb=mlb, require_no_errors=True):
    all_rows = load_prediction_jsonl_all(path)

    ok_rows = [row for row in all_rows if row.get("status") == "ok"]
    error_rows = [row for row in all_rows if row.get("status") == "error"]

    if require_no_errors and len(error_rows) > 0:
        raise ValueError(
            f"{path} contains {len(error_rows)} error rows. ")

    y_true_labels = [row["gold_labels"] for row in ok_rows]
    y_pred_labels = [row["predicted_labels"] for row in ok_rows]

    y_true = mlb.transform(y_true_labels)
    y_pred = mlb.transform(y_pred_labels)

    return {
        "path": path,
        "n_total_rows": len(all_rows),
        "n_examples": len(ok_rows),
        "n_error_rows": len(error_rows),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "avg_predicted_labels": np.mean([len(row["predicted_labels"]) for row in ok_rows]),
        "avg_gold_labels": np.mean([len(row["gold_labels"]) for row in ok_rows]),
        "avg_input_tokens": np.mean([row["input_tokens"] for row in ok_rows]),
        "avg_output_tokens": np.mean([row["output_tokens"] for row in ok_rows]),
        "avg_runtime_seconds": np.mean([row["wall_time_seconds"] for row in ok_rows])
    }

testing different values of the n-retrieved samples on the best strategy (topk+labels) - was chosen on n_retrieved = 5

In [ ]:
#for N_RETRIEVED = 3
TOPK3_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_topk3_predictions.jsonl"
run_ollama_validation_checkpointed(
    items=val_topk_items,
    output_path=TOPK3_VAL_OUTPUT,
    strategy_name="topk_3_examples_label_descriptions",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=val_gold_labels)

Already completed: 0
Remaining to run: 1000
Finished. Total wall time: 39.10 minutes


In [ ]:
#N_RETRIEVED = 1
TOPK1_VAL_OUTPUT = f"{OUTPUT_DIR}/AAPD_llama_OLLAMA_val_topk1_predictions.jsonl"
run_ollama_validation_checkpointed(
    items=val_topk_items,
    output_path=TOPK1_VAL_OUTPUT,
    strategy_name="topk_1_examples_label_descriptions",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=val_gold_labels)

Already completed: 0
Remaining to run: 1000
Finished. Total wall time: 30.96 minutes


In [ ]:
results = pd.DataFrame([
    evaluate_prediction_file(TOPK3_VAL_OUTPUT),
    evaluate_prediction_file(TOPK1_VAL_OUTPUT)])

results_path = f"{OUTPUT_DIR}/AAPD_llama_validation_top_3_1_results_comparison.csv"
results.to_csv(results_path, index=False)

results
#the f1-macro scores are lower than for the n = 5 - which is 0.52 on validation set

,path,n_total_rows,n_examples,n_error_rows,micro_f1,macro_f1,avg_predicted_labels,avg_gold_labels,avg_input_tokens,avg_output_tokens,avg_runtime_seconds
0,/content/drive/MyDrive/thesis_results/AAPD_LLA...,1000,1000,0,0.554622,0.488557,2.122,2.4,3043.291,18.653,2.340966
1,/content/drive/MyDrive/thesis_results/AAPD_LLA...,1000,1000,0,0.497010,0.447229,2.115,2.4,2623.876,18.840,1.852697
